<a href="https://colab.research.google.com/github/pramathi-sujay/Sound-Detection-Model/blob/main/AIISH_YAMNet_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q tensorflow tensorflow-hub tensorflow-io librosa soundfile


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 6.6 MB/s eta 0:00:00


In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_io as tfio
import numpy as np
import pandas as pd
import librosa
import os

/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl5mutex6unlockEv']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZNK10tensorflow4data11DatasetBase8FinalizeEPNS_15OpKernelContextESt8functionIFN4absl12lts_202

In [3]:
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

print("YAMNet Loaded Successfully!")

YAMNet Loaded Successfully!


In [4]:
print(type(yamnet))

<class 'tensorflow.python.saved_model.load.Loader._recreate_base_user_object.<locals>._UserObject'>


In [5]:
print(yamnet.signatures)

_SignatureMap({'serving_default': <ConcreteFunction (*, waveform: TensorSpec(shape=(None,), dtype=tf.float32, name='waveform')) -> Dict[['output_2', TensorSpec(shape=(None, 64), dtype=tf.float32, name='output_2')], ['output_1', TensorSpec(shape=(None, 1024), dtype=tf.float32, name='output_1')], ['output_0', TensorSpec(shape=(None, 521), dtype=tf.float32, name='output_0')]] at 0x7C92D952ABA0>})


In [6]:
import kagglehub

path = kagglehub.dataset_download("chrisfilo/urbansound8k")

print(path)

100%|██████████| 5.61G/5.61G [01:08<00:00, 87.7MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/chrisfilo/urbansound8k/versions/1


In [7]:
metadata = pd.read_csv(
    f"{path}/UrbanSound8K.csv"
)

metadata.head()


,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing


In [8]:
print(metadata.shape)
print(metadata.columns)

(8732, 8)
Index(['slice_file_name', 'fsID', 'start', 'end', 'salience', 'fold',
       'classID', 'class'],
      dtype='object')


In [9]:
metadata["file_path"] = metadata.apply(
    lambda row: f"{path}/fold{row['fold']}/{row['slice_file_name']}",
    axis=1
)

metadata[["slice_file_name", "class", "file_path"]].head()

,slice_file_name,class,file_path
0,100032-3-0-0.wav,dog_bark,/root/.cache/kagglehub/datasets/chrisfilo/urba...
1,100263-2-0-117.wav,children_playing,/root/.cache/kagglehub/datasets/chrisfilo/urba...
2,100263-2-0-121.wav,children_playing,/root/.cache/kagglehub/datasets/chrisfilo/urba...
3,100263-2-0-126.wav,children_playing,/root/.cache/kagglehub/datasets/chrisfilo/urba...
4,100263-2-0-137.wav,children_playing,/root/.cache/kagglehub/datasets/chrisfilo/urba...


In [10]:
import os

print(os.path.exists(metadata.iloc[0]["file_path"]))

True


In [11]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

metadata["label"] = label_encoder.fit_transform(metadata["class"])

metadata[["class","label"]].drop_duplicates().sort_values("label")

,class,label
22,air_conditioner,0
9,car_horn,1
1,children_playing,2
0,dog_bark,3
196,drilling,4
122,engine_idling,5
106,gun_shot,6
171,jackhammer,7
114,siren,8
94,street_music,9


In [12]:
train_df = metadata[metadata["fold"] <= 8].reset_index(drop=True)

val_df = metadata[metadata["fold"] == 9].reset_index(drop=True)

test_df = metadata[metadata["fold"] == 10].reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 7079
Validation: 816
Test: 837


In [13]:
train_files = train_df["file_path"].tolist()
train_labels = train_df["label"].tolist()

val_files = val_df["file_path"].tolist()
val_labels = val_df["label"].tolist()

test_files = test_df["file_path"].tolist()
test_labels = test_df["label"].tolist()

In [14]:
print(train_files[0])
print(train_labels[0])

/root/.cache/kagglehub/datasets/chrisfilo/urbansound8k/versions/1/fold5/100032-3-0-0.wav
3


In [15]:
import librosa
import numpy as np

def load_wav_16k_mono(filename):
    """
    Load an audio file as a mono waveform sampled at 16 kHz.
    """
    audio, sample_rate = librosa.load(
        filename,
        sr=16000,      # Resample to 16 kHz
        mono=True      # Convert to mono
    )

    # Convert to TensorFlow tensor
    audio = tf.convert_to_tensor(audio, dtype=tf.float32)

    return audio

In [16]:
audio = load_wav_16k_mono(train_files[0])

print(audio)
print(audio.shape)
print(audio.dtype)

tf.Tensor(
[-0.00300996 -0.00513961 -0.0046701  ... -0.00318873 -0.00204857
 -0.00052394], shape=(5081,), dtype=float32)
(5081,)
<dtype: 'float32'>


In [17]:
print(type(yamnet))

<class 'tensorflow.python.saved_model.load.Loader._recreate_base_user_object.<locals>._UserObject'>


In [18]:
!git clone https://github.com/tensorflow/models.git

Cloning into 'models'...
remote: Enumerating objects: 105594, done.
remote: Total 105594 (delta 0), reused 0 (delta 0), pack-reused 105594 (from 1)
Receiving objects: 100% (105594/105594), 644.03 MiB | 31.16 MiB/s, done.
Resolving deltas: 100% (76434/76434), done.


In [19]:
import os

yamnet_path = "models/research/audioset/yamnet"

print(os.listdir(yamnet_path))

['features.py', 'yamnet_visualization.ipynb', 'inference.py', 'yamnet.py', 'README.md', 'yamnet_test.py', 'yamnet_class_map.csv', 'export.py', 'params.py']


In [20]:
with open("models/research/audioset/yamnet/yamnet.py", "r") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

Total lines: 138


In [21]:
for i, line in enumerate(lines[:40]):
    print(f"{i+1}: {line}", end="")

1: # Copyright 2019 The TensorFlow Authors All Rights Reserved.
2: #
3: # Licensed under the Apache License, Version 2.0 (the "License");
4: # you may not use this file except in compliance with the License.
5: # You may obtain a copy of the License at
6: #
7: #     http://www.apache.org/licenses/LICENSE-2.0
8: #
9: # Unless required by applicable law or agreed to in writing, software
10: # distributed under the License is distributed on an "AS IS" BASIS,
11: # WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
12: # See the License for the specific language governing permissions and
13: # limitations under the License.
14: # ==============================================================================
15: 
16: """Core model definition of YAMNet."""
17: 
18: import csv
19: 
20: import numpy as np
21: import tensorflow as tf
22: from tf_keras import Model, layers
23: 
24: import features as features_lib
25: 
26: 
27: def _batch_norm(name, params):
28:   def _bn_la

In [22]:
for i, line in enumerate(lines[40:90], start=41):
    print(f"{i}: {line}", end="")

41:                            kernel_size=kernel,
42:                            strides=stride,
43:                            padding=params.conv_padding,
44:                            use_bias=False,
45:                            activation=None)(layer_input)
46:     output = _batch_norm('{}/conv/bn'.format(name), params)(output)
47:     output = layers.ReLU(name='{}/relu'.format(name))(output)
48:     return output
49:   return _conv_layer
50: 
51: 
52: def _separable_conv(name, kernel, stride, filters, params):
53:   def _separable_conv_layer(layer_input):
54:     output = layers.DepthwiseConv2D(name='{}/depthwise_conv'.format(name),
55:                                     kernel_size=kernel,
56:                                     strides=stride,
57:                                     depth_multiplier=1,
58:                                     padding=params.conv_padding,
59:                                     use_bias=False,
60:                                     activatio

In [23]:
for i, line in enumerate(lines[90:], start=91):
    print(f"{i}: {line}", end="")

91:     (_separable_conv, [3, 3], 1, 1024)
92: ]
93: 
94: 
95: def yamnet(features, params):
96:   """Define the core YAMNet mode in Keras."""
97:   net = layers.Reshape(
98:       (params.patch_frames, params.patch_bands, 1),
99:       input_shape=(params.patch_frames, params.patch_bands))(features)
100:   for (i, (layer_fun, kernel, stride, filters)) in enumerate(_YAMNET_LAYER_DEFS):
101:     net = layer_fun('layer{}'.format(i + 1), kernel, stride, filters, params)(net)
102:   embeddings = layers.GlobalAveragePooling2D()(net)
103:   logits = layers.Dense(units=params.num_classes, use_bias=True)(embeddings)
104:   predictions = layers.Activation(activation=params.classifier_activation)(logits)
105:   return predictions, embeddings
106: 
107: 
108: def yamnet_frames_model(params):
109:   """Defines the YAMNet waveform-to-class-scores model.
110: 
111:   Args:
112:     params: An instance of Params containing hyperparameters.
113: 
114:   Returns:
115:     A model accepting (num_samples

In [24]:
import sys

sys.path.append("models/research/audioset/yamnet")

In [25]:
import params
import yamnet

In [26]:
params = params.Params()

In [27]:
model = yamnet.yamnet_frames_model(params)

In [28]:
model.summary()

Model: "yamnet_frames"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None,)]                    0         []                            
                                                                                                  
 tf.compat.v1.shape (TFOpLa  (1,)                         0         ['input_1[0][0]']             
 mbda)                                                                                            
                                                                                                  
 tf.__operators__.getitem (  ()                           0         ['tf.compat.v1.shape[0][0]']  
 SlicingOpLambda)                                                                                 
                                                                                      

In [29]:
for i, output in enumerate(model.outputs):
    print(f"Output {i}: {output}")

Output 0: KerasTensor(type_spec=TensorSpec(shape=(None, 521), dtype=tf.float32, name=None), name='activation/Sigmoid:0', description="created by layer 'activation'")
Output 1: KerasTensor(type_spec=TensorSpec(shape=(None, 1024), dtype=tf.float32, name=None), name='global_average_pooling2d/Mean:0', description="created by layer 'global_average_pooling2d'")
Output 2: KerasTensor(type_spec=TensorSpec(shape=(None, 64), dtype=tf.float32, name=None), name='tf.math.log/Log:0', description="created by layer 'tf.math.log'")


In [30]:
print(model.weights[0].numpy().mean())

-0.0005967847


In [31]:
!wget -O models/research/audioset/yamnet/yamnet.h5 \
https://storage.googleapis.com/audioset/yamnet.h5

--2026-07-01 16:35:42--  https://storage.googleapis.com/audioset/yamnet.h5
Resolving storage.googleapis.com (storage.googleapis.com)... 192.178.129.207, 192.178.210.207, 142.251.189.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|192.178.129.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15296092 (15M) [application/octet-stream]
Saving to: ‘models/research/audioset/yamnet/yamnet.h5’

models/research/aud 100%[===================>]  14.59M  31.1MB/s    in 0.5s    

2026-07-01 16:35:42 (31.1 MB/s) - ‘models/research/audioset/yamnet/yamnet.h5’ saved [15296092/15296092]



In [32]:
import os

print(os.path.exists("models/research/audioset/yamnet/yamnet.h5"))

True


In [33]:
model.load_weights("models/research/audioset/yamnet/yamnet.h5")

In [34]:
print("Weights loaded successfully!")

Weights loaded successfully!


In [35]:
audio = load_wav_16k_mono(train_files[0])

predictions, embeddings, spectrogram = model(audio)

print(predictions.shape)
print(embeddings.shape)
print(spectrogram.shape)

(1, 521)
(1, 1024)
(96, 64)


In [36]:
scores = predictions.numpy()

top_class = np.argmax(scores.mean(axis=0))

print(top_class)

69


In [37]:
class_names = yamnet.class_names(
    "models/research/audioset/yamnet/yamnet_class_map.csv"
)

print(class_names[top_class])

Dog


In [38]:
import os

folders = [
    "utils",
    "checkpoints",
    "saved_models"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project structure created!")

Project structure created!


In [41]:
import librosa
import tensorflow as tf

def load_wav_16k_mono(filename):
    audio, _ = librosa.load(
        filename,
        sr=16000,
        mono=True
    )

    return tf.convert_to_tensor(audio, dtype=tf.float32)